In [2]:
import torch
from monai.data import ImageDataset, DataLoader
from monai.transforms import EnsureChannelFirst, Compose, Rand3DElastic, Resize, NormalizeIntensity, RandShiftIntensity
import monai
from torch.utils.tensorboard import SummaryWriter
import json
import os
from collections import OrderedDict

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Transforms from the MONAI tutorial
train_transforms = Compose([NormalizeIntensity(nonzero=True, channel_wise=True), 
                            EnsureChannelFirst(), 
                            Resize((96, 96, 96)),
                            Rand3DElastic(prob=0.5, sigma_range=(6, 8), magnitude_range=(10, 50),spatial_size=(96, 96, 96), padding_mode="border"),
                            RandShiftIntensity(prob=0.5, offsets=0.10)
])

val_transforms = Compose([NormalizeIntensity(nonzero=True, channel_wise=True), EnsureChannelFirst(), Resize((96, 96, 96))])

## 1. Load the locked-in splits
load_path = os.path.expanduser('~/Desktop/brain-math/deeplearn/GLM/GLM_kfold_splits.json')
with open(load_path, 'r') as f:
    saved_splits = json.load(f)

# 2. Select which fold you want to train right now
current_fold = "fold_1"  # Change this to "fold_2", "fold_3", etc., when ready
print(f"Loading data for {current_fold}...")

fold_data = saved_splits[current_fold]
train_images = fold_data["train_images"]
val_images = fold_data["val_images"]

# 3. Convert the saved integer labels (0 or 1) back into one-hot tensors for MONAI
train_labels = torch.as_tensor(fold_data["train_labels"], dtype=torch.long)
val_labels = torch.as_tensor(fold_data["val_labels"], dtype=torch.long)

# 4. Create your MONAI Datasets
train_ds = ImageDataset(image_files=train_images, labels=train_labels, transform=train_transforms)
val_ds = ImageDataset(image_files=val_images, labels=val_labels, transform=val_transforms)

# 5. Create DataLoaders
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=8, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=8, num_workers=8, pin_memory=torch.cuda.is_available())

model = monai.networks.nets.ViT(
    in_channels=1,
    img_size=(96, 96, 96),
    patch_size=(16, 16, 16), # Slices the 96x96x96 brain into 216 individual cubes
    proj_type='conv',        # Uses 3D convolutions to learn where each cube belongs in space
    classification=True,     # Forces the model to output a class label, not a segmentation mask
    num_classes=2            # Math Difficulty vs. Control
).to(device)

loss_function = torch.nn.CrossEntropyLoss()

# Replace your current optimizer with AdamW and add weight_decay
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)



# start a typical PyTorch training
best_metric = -1
best_metric_epoch = -1
epoch_loss_values = []
metric_values = []
writer = SummaryWriter(log_dir="runs/ViT_test")
max_epochs = 150

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)

patience = 30            # Stop after 20 validation checks without improvement
patience_counter = 0     # Tracks how long we've gone without a new best score

for epoch in range(max_epochs):
    print("-" * 10)
    print(f"epoch {epoch + 1}/{max_epochs}")
    model.train()
    epoch_loss = 0
    step = 0

    for batch_data in train_loader:
        step += 1
        inputs, labels = batch_data[0].to(device), batch_data[1].to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs[0], labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        epoch_len = len(train_ds) // train_loader.batch_size
        print(f"{step}/{epoch_len}, train_loss: {loss.item():.4f}")
        writer.add_scalar("train_loss", loss.item(), epoch_len * epoch + step)

    epoch_loss /= step
    epoch_loss_values.append(epoch_loss)
    print(f"epoch {epoch + 1} average loss: {epoch_loss:.4f}")


    model.eval()

    num_correct = 0.0
    metric_count = 0
    val_loss_sum = 0.0
    for val_data in val_loader:
        val_images, val_labels = val_data[0].to(device), val_data[1].to(device)
        with torch.no_grad():
            val_outputs = model(val_images)
            logits = val_outputs[0]
            val_loss = loss_function(logits, val_labels)
            val_loss_sum += val_loss.item()
            value = torch.eq(logits.argmax(dim=1), val_labels)
            metric_count += len(value)
            num_correct += value.sum().item()

    metric = num_correct / metric_count
    metric_values.append(metric)

    avg_val_loss = val_loss_sum / len(val_loader)

    if metric > best_metric:
        best_metric = metric
        best_metric_epoch = epoch + 1
        patience_counter = 0  # reset patience if it improves
        torch.save(model.state_dict(), "best_ViT_classification3d_array.pth")
        print("saved new best metric model")
    else:
        patience_counter += 1 # increment patience if it failed to improve
        print(f"No improvement. Patience: {patience_counter}/{patience}")

    print(f"Current epoch: {epoch+1} current accuracy: {metric:.4f} ")
    print(f"Best accuracy: {best_metric:.4f} at epoch {best_metric_epoch}")
    writer.add_scalar("val_accuracy", metric, epoch + 1)

    scheduler.step()

    if patience_counter >= patience:
        print(f"\nEarly stopping triggered at epoch {epoch + 1}!")
        print(f"Validation accuracy hasn't improved in {patience} epochs.")
        break

print(f"Training completed, best_metric: {best_metric:.4f} at epoch: {best_metric_epoch}")
writer.close()

Loading data for fold_1...
----------
epoch 1/150
1/24, train_loss: 0.9998
2/24, train_loss: 1.4676
3/24, train_loss: 1.3554
4/24, train_loss: 0.6273
5/24, train_loss: 0.8345
6/24, train_loss: 0.6568
7/24, train_loss: 0.7948
8/24, train_loss: 1.1274
9/24, train_loss: 0.6301
10/24, train_loss: 0.8232
11/24, train_loss: 0.6985
12/24, train_loss: 0.8389
13/24, train_loss: 0.6931
14/24, train_loss: 0.6253
15/24, train_loss: 0.8480
16/24, train_loss: 0.6931
17/24, train_loss: 0.6931
18/24, train_loss: 0.6931
19/24, train_loss: 0.7211
20/24, train_loss: 0.6339
21/24, train_loss: 0.6931
22/24, train_loss: 0.6919
23/24, train_loss: 0.6931
24/24, train_loss: 0.6952
epoch 1 average loss: 0.8012
saved new best metric model
Current epoch: 1 current accuracy: 0.3830 
Best accuracy: 0.3830 at epoch 1
----------
epoch 2/150
1/24, train_loss: 0.6225
2/24, train_loss: 0.7986
3/24, train_loss: 0.6931
4/24, train_loss: 0.6931
5/24, train_loss: 0.6931
6/24, train_loss: 0.6931
7/24, train_loss: 0.6931
8/24

KeyboardInterrupt: 